# NTX 18 — Official Agent Harness Gate-in-Path Audit

CPU scoring and integrity checks; official benchmark execution must be integrated in the GitHub repo first. No fabricated benchmark metrics.

## Official benchmarking rule

A qualifying AgentDojo / τ³ / BFCL-style result requires **tool-action interception inside the official harness**, real tool effects, the official scoring mechanism and paired starting state. An ungated native evaluation or mere adapter import is **not** a positive result. This notebook checks and analyses evidence exported by the Antigravity-built native adapter.

In [ ]:
from pathlib import Path
import sys, os, csv, json, hashlib, zipfile, subprocess, time
import pandas as pd
from datetime import datetime, timezone
KIT = Path('/content/NiyamTraceX_Paper_Final_Validation_Kit') if Path('/content').exists() else Path.cwd().parent
if not (KIT/'evidence').is_dir():
    here = Path.cwd()
    choices = [here, here.parent, Path('/mnt/data/NiyamTraceX_Paper_Final_Validation_Kit')]
    for candidate in choices:
        if (candidate/'evidence').is_dir(): KIT = candidate; break
if not (KIT/'evidence').is_dir():
    package = None
    for p in [Path.cwd()/'NiyamTraceX_Paper_Final_Validation_Kit.zip',Path('/content/NiyamTraceX_Paper_Final_Validation_Kit.zip')]:
        if p.exists(): package=p;break
    if package is None and 'google.colab' in sys.modules:
        from google.colab import files
        print('Upload NiyamTraceX_Paper_Final_Validation_Kit.zip (the full kit).')
        upload = files.upload()
        if 'NiyamTraceX_Paper_Final_Validation_Kit.zip' in upload:
            package = Path.cwd()/'NiyamTraceX_Paper_Final_Validation_Kit.zip'
    if package:
        with zipfile.ZipFile(package) as z:
            for part in z.infolist():
                target=(Path('/content') if Path('/content').exists() else Path.cwd())/part.filename
                if not str(target.resolve()).startswith(str((Path('/content') if Path('/content').exists() else Path.cwd()).resolve())):
                    raise RuntimeError('Unsafe archive path')
            z.extractall('/content' if Path('/content').exists() else '.')
        KIT = (Path('/content') if Path('/content').exists() else Path.cwd())/'NiyamTraceX_Paper_Final_Validation_Kit'
OUT = Path(os.getenv('NTX_PAPER_OUT',str(KIT/'outputs')))
OUT.mkdir(parents=True,exist_ok=True)
print('Kit:',KIT,'\nOutput:',OUT)
assert (KIT/'evidence').is_dir(), 'Evidence not found. Upload the full kit ZIP.'
def dump(obj,p):
    Path(p).parent.mkdir(parents=True,exist_ok=True)
    Path(p).write_text(json.dumps(obj,indent=2,default=str),encoding='utf-8')
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

In [ ]:
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import binomtest
import numpy as np
OFFICIAL=Path(os.getenv('NTX_OFFICIAL_BENCHMARK_CSV',str(KIT/'evidence'/'official_gate_in_path_results.csv')))
required=['official_suite','official_suite_version','official_task_id','variant','model_id','repo_commit',
          'initial_state_sha256','official_scorer_used','tool_hook_intercepted','executor_is_native',
          'task_success','unsafe_effect','event_log_sha256','final_state_sha256']
if not OFFICIAL.is_file():
    pd.DataFrame(columns=required).to_csv(OUT/'18_OFFICIAL_EXPORT_SCHEMA.csv',index=False)
    print('PENDING: no official gate-in-path export. Antigravity must integrate and run official harness.')
    scored=pd.DataFrame(columns=required)
else:
    scored=pd.read_csv(OFFICIAL,keep_default_na=False,dtype=str)
    assert set(required).issubset(scored.columns),sorted(set(required)-set(scored.columns))
    assert not scored.empty,'Empty official results'
    assert scored.variant.isin(['gated','ungated']).all()
    assert scored.official_suite.isin(['AgentDojo','tau3','tau2','BFCL']).all()
    assert scored.official_scorer_used.eq('True').all()
    assert scored.executor_is_native.eq('True').all()
    assert scored.tool_hook_intercepted.eq('True').all()
    assert scored.task_success.isin(['True','False']).all() and scored.unsafe_effect.isin(['True','False']).all()
    assert scored.repo_commit.str.fullmatch('[0-9a-f]{40}').all()
    for col in ['initial_state_sha256','event_log_sha256','final_state_sha256']:
        assert scored[col].str.fullmatch('[0-9a-f]{64}').all(),col
    assert not scored.duplicated(['official_suite','official_suite_version','official_task_id','model_id','variant']).any()

In [ ]:
qualifying=[];blockers=[]
if len(scored):
    key=['official_suite','official_suite_version','official_task_id','model_id']
    raw=scored[scored.variant=='ungated'].set_index(key)
    gate=scored[scored.variant=='gated'].set_index(key)
    if not raw.index.is_unique or not gate.index.is_unique:raise ValueError('Duplicate task keys')
    if not raw.index.equals(gate.index):
        assert set(raw.index)==set(gate.index),'Gated/ungated task IDs differ'
        raw=raw.sort_index();gate=gate.sort_index()
    if any(raw.initial_state_sha256!=gate.initial_state_sha256):raise RuntimeError('Starting states not identical')
    if any(raw.repo_commit!=gate.repo_commit):raise RuntimeError('Different code revisions')
    pairs=raw.reset_index()[key].copy()
    for col in ['task_success','unsafe_effect']:
        pairs[col+'_raw']=raw[col].eq('True').to_numpy()
        pairs[col+'_gate']=gate[col].eq('True').to_numpy()
    pairs.to_csv(OUT/'18_official_paired_tasks.csv',index=False)
    for (suite,ver,model),d in pairs.groupby(['official_suite','official_suite_version','model_id']):
        b=int((d.task_success_raw & ~d.task_success_gate).sum())
        c=int((~d.task_success_raw & d.task_success_gate).sum())
        n=len(d);k=int(d.unsafe_effect_gate.sum())
        qualifying.append({'suite':suite,'suite_version':ver,'model':model,'n_paired_tasks':n,
          'raw_success':float(d.task_success_raw.mean()),'gated_success':float(d.task_success_gate.mean()),
          'raw_unsafe':int(d.unsafe_effect_raw.sum()),'gated_unsafe':k,
          'gated_unsafe_upper95':float(proportion_confint(k,n,alpha=.05,method='beta')[1]),
          'exact_mcnemar_success_p':float(binomtest(min(b,c),b+c,.5).pvalue) if b+c else 1.0})
    pd.DataFrame(qualifying).to_csv(OUT/'18_official_summary.csv',index=False)
else:blockers.append('No official task-level native gate-in-path run')
result={'status':'TASK_LEVEL_EVIDENCE_CANDIDATE_NEEDS_TRACE_AUDIT' if qualifying else 'BLOCKED',
        'cohorts':qualifying,'blockers':blockers,'not_claimable':['official score until logs and scorer provenance verified','zero population risk from zero observed events']}
dump(result,OUT/'18_OFFICIAL_GATE_REPORT.json')
print(json.dumps(result,indent=2)[:5500])